In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# fix the path and read the csv file
# i use os.path.join to avoid any directory errors
file_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(file_path)
print("data loded corectly now!")

In [ ]:
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# food distribution (target variable)
# ploting Delivery_Time to see if there is outliers
# i use hist plot with orange color
plt.figure(figsize=(8, 4))

# Fixed: Changed 'delivery_time' to 'Delivery_Time' to match your data
plt.hist(df['Delivery_Time'], bins=30, color='orange', edgecolor='black')

plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery Time (min)")
plt.ylabel("count")
plt.show()

In [ ]:
# removing Order_ID column from the data because its not useful
df = df.drop('Order_ID', axis=1)

In [ ]:
# 2. Do we have missing values?
# checking for nulls in columns
# i will fill them with mean or just drop, lets drop to be safe
print("Missing values before:", df.isnull().sum().sum())
df = df.dropna()

In [ ]:
# 3. Do we have duplicate samples?
# finding if there is any duplicated rows

print("Duplicated rows found:", df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
# 4. Do we have categorical columns?

# this is better for the modle as a bonus
df = pd.get_dummies(df)
print("Encoding is finish")

In [ ]:

#.5
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

# scaling all features now
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# puting it back to dataframe
X = pd.DataFrame(X_scaled, columns=X.columns)
print("y is now defined and data is scaled!")

In [ ]:
# 6. Is the target imbalanced?

print("target mean is:", y.mean())

In [ ]:
# separating the data
# delivery_time is what we want to predict target
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

print("X shape is:", X.shape)
print("Target y is ready!")

In [ ]:
# i will use KFold because this is regression task
# stratified is only for classification labels
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

# starting the loop for training

for train_idx, test_idx in kf.split(X):
    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # define the modle Random Forest
    # i will put 100 estimators
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

    # fiting the data now
    rf_model.fit(X_train_f, y_train_f)

    # making prediction to check performance
    predictions = rf_model.predict(X_test_f)

    # calculating MAE only as requested in task 4
    fold_mae = mean_absolute_error(y_test_f, predictions)
    mae_scores.append(fold_mae)
    print(f"one fold MAE score: {fold_mae:.4f}")

# print the final resault
print("\n--- Final Results ---")
print("Average MAE for all folds is:", np.mean(mae_scores))

In [ ]:
# Task 1: Write your code here:


importances = rf_model.feature_importances_
features = X.columns

# ploting it to see the result
plt.figure(figsize=(10, 6))
plt.barh(features, importances, color='skyblue')
plt.title("Feature Importance - What makes delivery fast?")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

In [ ]:
# Task 2: Write your code here:
# ploting the predictions to see the distribution

plt.figure(figsize=(8, 5))
plt.hist(predictions, bins=20, color='lightgreen', edgecolor='black')
plt.title("Histogram of Predicted Delivery Times")
plt.xlabel("Predicted Minutes")
plt.ylabel("Frequency")
plt.show()



In [ ]:
from catboost import CatBoostRegressor

# kfold setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train_e, X_test_e = X.iloc[train_idx], X.iloc[test_idx]
    y_train_e, y_test_e = y.iloc[train_idx], y.iloc[test_idx]

    # traning 2 models
    model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
    model_cb = CatBoostRegressor(iterations=100, silent=True)

    model_rf.fit(X_train_e, y_train_e)
    model_cb.fit(X_train_e, y_train_e)

    # average the predictions
    p1 = model_rf.predict(X_test_e)
    p2 = model_cb.predict(X_test_e)
    final_p = (p1 + p2) / 2

    # calc mae
    m = mean_absolute_error(y_test_e, final_p)
    ensemble_mae_scores.append(m)

# results
print(f"Final Ensemble MAE: {np.mean(ensemble_mae_scores):.4f}")